# Knife-edge profile: derivative of `weird_two_peak_mode_2.txt`

This is a second measurement of the same mode as `derivada_perfil_dos_picos.ipynb`, using the same method.
The beam profile along the knife's travel is $I(t) = dP/dt$. This mode shows two lobes.

**Caveat:** the knife was moved by hand, so its speed is neither known nor constant. The x axis is time,
and the positions and widths are only qualitative.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.signal import savgol_filter
from scipy.optimize import curve_fit

# PM100D log: 2 header lines, then "dd/mm/yyyy hh:mm:ss,fff \t value \t W" (decimal comma)
t, P = [], []
with open('weird_two_peak_mode_2.txt', encoding='latin-1') as fh:
    for line in fh.read().splitlines()[2:]:
        parts = line.split('\t')
        if len(parts) < 2:
            continue
        t.append(datetime.strptime(parts[0].strip(), '%d/%m/%Y %H:%M:%S,%f').timestamp())
        P.append(float(parts[1].replace(',', '.')))
t = np.array(t) - t[0]          # s
P = np.array(P) * 1e3           # mW
t_raw, P_raw = t.copy(), P.copy()
print(f'{len(t)} points')

## Artifact 1: the power meter switching range

This is the same problem as in the other files. The meter is on `Range Auto`, and at about 4.7 mW (t ≈ 14 s) it
switches range: one reading jumps (+0.39 mW), then nothing is logged for 0.36 s. As before, the mismatch across the
gap is **lost time**, about 0.2 s.

The slope changes quickly right at this switch (the second lobe is starting), so fitting a straight line on each side
isn't reliable here. Instead we fit **one cubic across the gap**, with the time shift of the points after the gap as a
free parameter, and keep the shift that fits best. The same method gives 0.18–0.23 s in all three files.

The fix:
1. drop the jumped reading,
2. shift the timestamps after the switch forward by that amount.

## Artifact 2: drift before the knife moves

For the first ~4 s the power falls steadily (1.60 → 1.36 mW) before the knife starts uncovering the beam, which shows
up as a negative derivative. It isn't part of the beam profile, so we cut it.

To avoid all this next time, **fix the range manually** before scanning.

In [ ]:
gap = np.argmax(np.diff(t))                  # index right before the dead time
t_switch = t[gap]
print(f'dead time of {t[gap+1] - t[gap]:.3f} s at t = {t_switch:.2f} s')

t, P = np.delete(t, gap), np.delete(P, gap)  # the jumped reading, logged just before the dead time

near = (t > t_switch - 1) & (t < t_switch + 1.4)
after = t[near] > t_switch

def misfit(shift):
    x = t[near] + shift * after
    return np.sum((P[near] - np.polyval(np.polyfit(x, P[near], 3), x))**2)

shifts = np.arange(-0.3, 0.6, 0.002)
lost = shifts[np.argmin([misfit(s) for s in shifts])]
t[t > t_switch] += lost
print(f'time lost in the range switch: {lost:.3f} s (added back)')

# before ~4 s the knife isn't moving yet (the power just drifts down),
# after ~33 s the beam is fully uncovered
t_min, t_max = 4, 33
keep = (t > t_min) & (t < t_max)
t, P = t[keep], P[keep]

## Savitzky–Golay derivative

In [ ]:
dt = 0.02

def sg_deriv(tt, PP, win_s, order=3):
    tu = np.arange(tt[0], tt[-1], dt)
    n = int(round(win_s / dt)) | 1   # window length has to be odd
    return tu, savgol_filter(np.interp(tu, tt, PP), n, order, deriv=1, delta=dt)

fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax[0].plot(t_raw, P_raw, '.', color='0.7', ms=2, label='raw')
ax[0].plot(t, P, 'k.', ms=2, label='corrected + cut')
ax[0].axvline(t_switch, color='r', ls=':', lw=1)
ax[0].set_ylabel('P [mW]')
ax[0].legend()
tu_raw, d_raw = sg_deriv(t_raw, P_raw, 1.0)
ax[1].plot(tu_raw, d_raw, color='0.7', lw=1, label='raw, SG 1 s')
for win in [0.6, 1.0, 1.5]:
    ax[1].plot(*sg_deriv(t, P, win), lw=1, label=f'corrected, SG {win} s')
ax[1].axvline(t_switch, color='r', ls=':', lw=1)
ax[1].set_xlabel('t [s]')
ax[1].set_ylabel('dP/dt [mW/s]')
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

tu, dPdt = sg_deriv(t, P, 1.0)
print(f'area under dP/dt = {np.trapezoid(dPdt, tu):.3f} mW   vs   step = {P[-30:].mean() - P[:30].mean():.3f} mW')

## Two-lobe fit

The second lobe has the same long tail as the fundamental: a sharp rise followed by a slow decay. That shape is
most likely the hand slowing down near the end of the scan. So we compare two fits:
- two plain Gaussians,
- a Gaussian plus a split Gaussian (different widths on each side) for the second lobe.

The two plain Gaussians miss the dip between the lobes (around 12 s). The Gaussian + split Gaussian reproduces it
(rms residual 0.034 vs 0.046 mW/s), so it's the better description. The power split between the lobes depends
on the model, though (lobe 2 carries 2–6× the power of lobe 1), because the lobes overlap a lot.

Compared with the first measurement (`derivada_perfil_dos_picos.ipynb`), the shape is the same: a weaker first
lobe, then a stronger second lobe with a long tail. So the two-lobe structure is reproducible. The total power
is about half (9.5 vs 19.6 mW), and the scan took longer.

In [ ]:
def gauss(t, a, t0, w):
    return a * np.exp(-2 * (t - t0)**2 / w**2)

def split_gauss(t, a, t0, wl, wr):
    return a * np.exp(-2 * (t - t0)**2 / np.where(t < t0, wl, wr)**2)

# no constant baseline: dP/dt is ~0 before the knife moves, and a free constant
# just soaks up the slow tail (it stole ~3 mW of the lobes' power)
def two_gauss(t, a1, t1, w1, a2, t2, w2):
    return gauss(t, a1, t1, w1) + gauss(t, a2, t2, w2)

def gauss_split(t, a1, t1, w1, a2, t2, wl, wr):
    return gauss(t, a1, t1, w1) + split_gauss(t, a2, t2, wl, wr)

p2, c2 = curve_fit(two_gauss, tu, dPdt, p0=[0.45, 9.5, 4, 0.85, 16.5, 8])
ps, cs = curve_fit(gauss_split, tu, dPdt, p0=[0.45, 9.5, 4, 0.85, 16.5, 5, 9])

def power(a, w):   # area of a*exp(-2(t-t0)^2/w^2)
    return a * w * np.sqrt(np.pi / 2)

e2, es = np.sqrt(np.diag(c2)), np.sqrt(np.diag(cs))
print(f'two gaussians          (rms residual {np.std(dPdt - two_gauss(tu, *p2)):.3f} mW/s)')
print(f'  lobe 1: t0 = {p2[1]:.2f} ± {e2[1]:.2f} s,  w = {p2[2]:.2f} ± {e2[2]:.2f} s,  power = {power(p2[0], p2[2]):.2f} mW')
print(f'  lobe 2: t0 = {p2[4]:.2f} ± {e2[4]:.2f} s,  w = {p2[5]:.2f} ± {e2[5]:.2f} s,  power = {power(p2[3], p2[5]):.2f} mW')
print(f'gaussian + split       (rms residual {np.std(dPdt - gauss_split(tu, *ps)):.3f} mW/s)')
print(f'  lobe 1: t0 = {ps[1]:.2f} ± {es[1]:.2f} s,  w = {ps[2]:.2f} ± {es[2]:.2f} s,  power = {power(ps[0], ps[2]):.2f} mW')
print(f'  lobe 2: t0 = {ps[4]:.2f} ± {es[4]:.2f} s,  w = {ps[5]:.2f} / {ps[6]:.2f} s,  '
      f'power = {power(ps[3], (ps[5] + ps[6]) / 2):.2f} mW')

fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
for axis, f, p, title in [(ax[0], two_gauss, p2, 'two gaussians'), (ax[1], gauss_split, ps, 'gaussian + split gaussian')]:
    axis.plot(tu, dPdt, 'k-', lw=1, label='dP/dt (SG 1 s)')
    axis.plot(tu, f(tu, *p), 'r-', label=title)
    axis.plot(tu, gauss(tu, *p[0:3]), '--', lw=1)
    axis.plot(tu, f(tu, *p) - gauss(tu, *p[0:3]), '--', lw=1)
    axis.set_ylabel('dP/dt [mW/s]')
    axis.legend()
ax[1].set_xlabel('t [s]')
plt.tight_layout()
plt.show()